# 实践项目 01：MRI 脑肿瘤图像分割

这份 Notebook 从配对的二维 MRI 切片和肿瘤 mask 开始，完成数据核对、患者级划分、三通道同步预处理、轻量 U-Net、前景感知采样、训练、阈值选择和测试评价。源 TIFF 的三个已配准通道共同作为模型输入，mask 是训练和评价时表示肿瘤区域的目标。

Kaggle Notebook 是本项目的首选实践入口。打开公开 Notebook 后，点击“复制并编辑”保存到自己的账户，再按顺序运行单元格；下载到电脑运行是补充方式。每个任务都写明了输入、输出和需要填写的位置。

生成的图像和 JSON 只用于自己的观察和复盘。

**学生需要填写或修改的位置：** 带有 `TODO`、`None` 占位或“你的记录”的区域。先看 shape、输入输出说明和断言，再填写当前数据产生的结果，不要把参考数字写死。

## 任务总览

1. 查找 MRI 与 mask 并确认一一配对。
2. 统计患者数量、阳性 mask 数量和空 mask 比例。
3. 按患者划分训练集、验证集和测试集。
4. 对三个 MRI 通道分别归一化，训练集只增加水平翻转。
5. 补全 Dice 和 U-Net 卷积块。
6. 完成训练更新并保存训练曲线。
7. 只用验证集选择阈值，再在测试集展示预测结果。


## 需要保存的结果

- `task1_data_check.png`：一张 MRI、对应 mask 和叠加图。
- `task1_training_curve.png`：训练/验证损失和 Dice 曲线。
- `task1_prediction.png`：测试图像、真实 mask 和预测 mask。
- `task1_result.json`：数据规模、阈值、指标和空 mask 统计。

这些文件由 Notebook 自动写入 `OUT` 指向的工作目录，用于自己的观察和复盘。


In [ ]:
# 输入：无；输出：数据读取、训练和结果保存所需的库与路径。
from pathlib import Path  # 导入路径工具
import json, random, re  # 导入 JSON、随机数和正则表达式工具
import numpy as np  # 导入数组计算工具
import matplotlib.pyplot as plt  # 导入绘图工具
from PIL import Image  # 导入图像读取工具
import torch  # 导入 PyTorch
import torch.nn as nn  # 导入神经网络模块
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler  # 导入数据集、批处理和加权采样器
from sklearn.model_selection import GroupShuffleSplit  # 导入患者级分组划分

SEED=42  # 固定随机状态
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)  # 同步 Python、NumPy 和 PyTorch 随机状态
CUDA_OK=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0]>=7  # 排除当前 PyTorch 不支持的旧 GPU
DEVICE=torch.device('cuda' if CUDA_OK else 'cpu')  # 使用兼容 GPU，否则安全回退到 CPU
OUT=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()/'kydw_outputs'  # 设置输出目录
OUT.mkdir(parents=True,exist_ok=True)  # 创建输出目录
print('device:',DEVICE)  # 显示运行设备


## 1. 查找并配对图像与 mask

**输入：** 推荐挂载 `lgg-mri-segmentation` 数据集。每个 TIFF 含三个已配准 MRI 通道，MRI 文件与 mask 文件在文件名上只相差 `_mask`，患者标识取自 mask 的上级文件夹。

**处理：** 代码按路径排序后遍历数据目录，把每个 mask 与去掉 `_mask` 后仍然存在的 MRI 配成一项 `(image_path, mask_path, patient_id)`。

**输出：** `pairs` 列表和 `paired slices` 数量。完整数据应得到 3929 对；断言用来检查数据路径和文件名规则。


In [ ]:
# 输入：数据集根目录；输出：每项 (image_path, mask_path, patient_id) 的 pairs。
kaggle_root=Path('/kaggle/input')  # Kaggle 数据集挂载根目录
local_root=Path.cwd()/'_codex_state'/'lgg_full'/'extracted'/'kaggle_3m'  # 本地复核数据目录
ROOT=kaggle_root if kaggle_root.exists() and any(kaggle_root.rglob('*_mask.tif')) else local_root  # 选择实际可用的数据根目录
all_tif=sorted(p for p in ROOT.rglob('*') if p.is_file() and p.suffix.lower() in {'.tif','.tiff','.png'} and '__MACOSX' not in p.parts and not p.name.startswith('._'))  # 排序后查找图像文件
mask_paths=sorted(p for p in all_tif if p.stem.lower().endswith('_mask'))  # 查找 mask
pairs=[]; seen=set()  # 保存配对结果并去除 Kaggle 挂载中的重复副本
for m in mask_paths:  # 逐个处理 mask
    img=m.with_name(re.sub(r'_mask(?=\.[^.]+$)', '', m.name, flags=re.IGNORECASE))  # 得到对应 MRI 路径
    if img.exists():  # 只保留真实配对
        patient=m.parent.name  # 使用患者文件夹作为分组键
        key=(patient,m.name)  # 同一患者的同名切片只保留一份
        if key in seen: continue  # 跳过 Kaggle 数据目录中的重复副本
        seen.add(key)  # 记录已使用的患者—切片组合
        pairs.append((img,m,patient))  # 保存 MRI、mask 和患者标识
print('paired slices:',len(pairs))  # 显示配对切片数量
assert len(pairs)==3929, f'期望 3929 对 MRI/mask，实际找到 {len(pairs)}；请检查数据集挂载。'  # 检查完整数据集


## 任务 1：完成数据核对

这一步先确认输入数据真的可以使用。阳性 mask 表示其中至少有一个前景像素；空 mask 表示整张 mask 没有前景。

**学生需要填写：** 根据 `pairs` 中的患者标识和每个 mask 的实际像素，计算唯一患者数 `patient_count`、阳性 mask 数 `positive_masks` 和空 mask 比例 `empty_ratio`。

**输入与输出：** 输入是 `pairs`；输出是三个标量，以及一张 MRI、mask 和 overlay 图。单张 `img` 与 `mask` 都应是 `[H, W]`，尺寸必须一致。

**检查：** 统计只能来自当前文件；运行后应打印三个统计量，并保存 `task1_data_check.png`。


In [ ]:
# ===== 请在此处完成：项目01·任务1 患者与 mask 数据统计（开始） =====
# TODO 1
patient_count = None  # 保存当前步骤使用的中间结果
positive_masks = None  # 保存当前步骤使用的中间结果
empty_ratio = None  # 保存当前步骤使用的中间结果
# ===== 请在此处完成：项目01·任务1 患者与 mask 数据统计（结束） =====

sample_img_path, sample_mask_path, _ = pairs[len(pairs)//2]  # 保存当前步骤使用的中间结果
img=np.array(Image.open(sample_img_path).convert('L'))  # 读取本任务需要的数据
mask=np.array(Image.open(sample_mask_path).convert('L'))>0  # 读取本任务需要的数据
print(patient_count, positive_masks, empty_ratio, img.shape, mask.shape)  # 显示便于检查的关键信息

fig,ax=plt.subplots(1,3,figsize=(10,3))  # 绘制当前步骤的结果图
ax[0].imshow(img,cmap='gray'); ax[0].set_title('MRI')  # 绘制当前步骤的结果图
ax[1].imshow(mask,cmap='gray'); ax[1].set_title('mask')  # 绘制当前步骤的结果图
ax[2].imshow(img,cmap='gray'); ax[2].imshow(mask,alpha=.4,cmap='viridis'); ax[2].set_title('overlay')  # 绘制当前步骤的结果图
for a in ax:a.axis('off')  # 逐批或逐样本执行当前步骤
plt.tight_layout(); plt.savefig(OUT/'task1_data_check.png',dpi=160); plt.show()  # 绘制当前步骤的结果图

## 2. 患者级划分

同一患者可能有多张切片。如果把同一患者的切片拆到不同集合，测试评价会偏乐观。因此这里先按 `patient_id` 分组，再划分训练、验证和测试集，不截断切片列表。

**输入：** `pairs` 中的图像、mask 和患者标识。

**输出：** `tr_idx`、`va_idx`、`test_idx` 三组切片索引，以及三个互不重叠的患者集合。交集断言必须通过。


In [ ]:
# 输入：pairs 中的患者分组；输出：患者不重叠的 tr_idx、va_idx、test_idx。
groups=np.array([item[2] for item in pairs]); idx=np.arange(len(pairs))  # 建立患者分组和切片索引
gss=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED)  # 固定 20% 患者用于测试
train_idx,test_idx=next(gss.split(idx,groups=groups))  # 先划分训练开发集和测试集
train_groups=groups[train_idx]  # 读取训练开发集患者
gss2=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED)  # 从训练开发集中固定验证患者
tr_rel,va_rel=next(gss2.split(train_idx,groups=train_groups))  # 获得相对索引
tr_idx=train_idx[tr_rel]; va_idx=train_idx[va_rel]  # 转回完整切片索引
assert not (set(groups[tr_idx]) & set(groups[va_idx]) | set(groups[tr_idx]) & set(groups[test_idx]) | set(groups[va_idx]) & set(groups[test_idx]))  # 检查患者无交叉
print(len(tr_idx),len(va_idx),len(test_idx),len(set(groups[tr_idx])),len(set(groups[va_idx])),len(set(groups[test_idx])))  # 显示切片和患者数量


In [ ]:
# 输入：pairs 和三组切片索引；输出：训练、验证、测试 DataLoader。
def normalize_channels(array):  # 对每个 MRI 通道分别进行百分位归一化
    array=array.astype(np.float32); output=np.zeros_like(array,dtype=np.float32)  # 准备输出数组
    brain=array.max(axis=2)>0  # 用非零区域近似脑区
    for channel in range(array.shape[2]):  # 分通道处理
        values=array[...,channel][brain]  # 只在脑区估计强度范围
        low,high=np.percentile(values,[1,99])  # 计算 1% 和 99% 分位数
        if high<=low: high=low+1  # 避免除零
        output[...,channel]=np.clip((array[...,channel]-low)/(high-low),0,1)  # 裁剪并缩放到 0—1
    return output  # 返回三通道图像

class MRIDataset(Dataset):  # 定义 MRI 分割数据集
    def __init__(self,pairs,indices,size=128,augment=False):  # 保存切片索引和设置
        self.pairs=pairs; self.indices=np.asarray(indices); self.size=size; self.augment=augment  # 记录数据
    def __len__(self): return len(self.indices)  # 返回样本数
    def __getitem__(self,position):  # 读取一张 MRI 和 mask
        index=int(self.indices[position]); ip,mp,pid=self.pairs[index]  # 找到文件路径和患者
        image=Image.open(ip).convert('RGB').resize((self.size,self.size),Image.Resampling.BILINEAR)  # 读取三个已配准通道
        mask=Image.open(mp).convert('L').resize((self.size,self.size),Image.Resampling.NEAREST)  # 最近邻缩放 mask
        x=normalize_channels(np.asarray(image))  # 分通道归一化
        y=(np.asarray(mask)>0).astype(np.float32)  # 转为二值 mask
        if self.augment and random.random()<.5:  # 训练集随机水平翻转
            x=np.fliplr(x).copy(); y=np.fliplr(y).copy()  # 同步翻转图像和 mask
        return torch.from_numpy(np.moveaxis(x,-1,0)),torch.from_numpy(y[None]),pid  # 返回 [3,H,W] 和 [1,H,W]

train_ds=MRIDataset(pairs,tr_idx,augment=True); val_ds=MRIDataset(pairs,va_idx); test_ds=MRIDataset(pairs,test_idx)  # 建立三组数据
train_positive=np.array([bool(np.asarray(Image.open(pairs[i][1]).convert('L')).max()>0) for i in tr_idx])  # 读取训练切片是否含肿瘤
sample_weights=np.where(train_positive,3.0,.65)  # 提高阳性切片被抽到的概率，同时保留空 mask
sampler=WeightedRandomSampler(sample_weights,num_samples=1200,replacement=True,generator=torch.Generator().manual_seed(SEED))  # 每轮固定抽取 1200 张
train_loader=DataLoader(train_ds,batch_size=32,sampler=sampler,num_workers=2)  # 建立训练批次
val_loader=DataLoader(val_ds,batch_size=32,shuffle=False,num_workers=2)  # 建立验证批次
test_loader=DataLoader(test_ds,batch_size=32,shuffle=False,num_workers=2)  # 建立测试批次


## 任务 2：补全 Dice

Dice 用预测前景与真实前景的重叠程度评价分割结果。`prob` 是模型经过 sigmoid 后的概率图，`target` 是 0/1 mask；概率大于等于 `threshold` 的像素属于预测前景。

**学生需要填写：** 在 `dice_score` 的标记区域中完成阈值化、交集、预测面积和真实面积的计算，并返回 batch 内样本 Dice 的平均值。

**输入与输出：** `prob`、`target` 和 `pred` 的 shape 都是 `[batch, 1, H, W]`；沿空间维度求和后，每个样本得到一个 Dice，函数最后输出一个标量。

**检查：** 使用给定的 `threshold`、`eps` 和空间维度；`soft_dice_score` 已提供给训练损失使用。后面的训练应能得到有限的 Dice。


In [ ]:
def dice_score(prob,target,threshold=.5,eps=1e-6):  # 定义可重复调用的计算步骤
# ===== 请在此处完成：项目01·任务2 阈值化 Dice（开始） =====
    # TODO 2：阈值化、计算交集与两侧面积
    return None  # 返回当前步骤的计算结果
# ===== 请在此处完成：项目01·任务2 阈值化 Dice（结束） =====

def soft_dice_score(prob,target,eps=1e-6):  # 定义可重复调用的计算步骤
    dims=tuple(range(1,prob.ndim))  # 计算模型输出或预测概率
    inter=(prob*target).sum(dims)  # 计算模型输出或预测概率
    return (2*inter+eps)/(prob.sum(dims)+target.sum(dims)+eps)  # 返回当前步骤的计算结果

## 任务 3：补全 U-Net 卷积块

`DoubleConv` 是 U-Net 中重复使用的小模块：两次 `3×3` 卷积，每次卷积后进行 GroupNorm 和 SiLU。`padding=1` 让高度和宽度保持不变，便于后面与跳跃连接拼接。

**输入与输出：** 输入是 `[batch, cin, H, W]`，输出是 `[batch, cout, H, W]`。模型读取三个 MRI 通道，并使用 8、16、32 个通道构成可在普通环境运行的两层 U-Net。


In [ ]:
class DoubleConv(nn.Module):  # 定义 U-Net 重复卷积块
    def __init__(self,cin,cout):  # 设置输入和输出通道
        super().__init__()  # 初始化父类
# ===== 请在此处完成：项目01·任务3 DoubleConv 卷积块（开始） =====
        # TODO 3：两次 3×3 卷积，每次接 GroupNorm 和 SiLU
        self.block = None  # 保存待完成的卷积块
# ===== 请在此处完成：项目01·任务3 DoubleConv 卷积块（结束） =====
    def forward(self,x): return self.block(x)  # 执行卷积块

class TinyUNet(nn.Module):  # 定义两层轻量 U-Net
    def __init__(self):  # 建立网络层
        super().__init__()  # 初始化父类
        self.e1=DoubleConv(3,8); self.p1=nn.MaxPool2d(2)  # 第一层编码
        self.e2=DoubleConv(8,16); self.p2=nn.MaxPool2d(2)  # 第二层编码
        self.b=DoubleConv(16,32)  # 瓶颈层
        self.u2=nn.ConvTranspose2d(32,16,2,2); self.d2=DoubleConv(32,16)  # 第二层解码
        self.u1=nn.ConvTranspose2d(16,8,2,2); self.d1=DoubleConv(16,8)  # 第一层解码
        self.out=nn.Conv2d(8,1,1)  # 输出一个肿瘤概率通道
    def forward(self,x):  # 前向传播
        e1=self.e1(x); e2=self.e2(self.p1(e1)); b=self.b(self.p2(e2))  # 编码
        d2=self.d2(torch.cat([self.u2(b),e2],1))  # 拼接第二层跳跃连接
        d1=self.d1(torch.cat([self.u1(d2),e1],1))  # 拼接第一层跳跃连接
        return self.out(d1)  # 返回 logits

model=TinyUNet().to(DEVICE)  # 把模型放到运行设备
print('parameters:',sum(p.numel() for p in model.parameters()))  # 显示参数量


## 任务 4：补全一次训练更新

训练分支每批先清空旧梯度，再计算 logits 和组合损失，反向传播、裁剪梯度并更新参数。验证分支只计算损失和阳性切片软 Dice。每轮使用前景感知采样固定抽取 1200 张切片，完整训练预算为 14 轮。

**输入与输出：** `x` 是 `[batch, 3, H, W]`，`y` 和 `logits` 是 `[batch, 1, H, W]`；输出为训练/验证损失与阳性切片软 Dice。


In [ ]:
bce=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(6.0,device=DEVICE))  # 提高肿瘤像素在 BCE 中的权重
opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)  # 配置 AdamW
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=14)  # 逐轮降低学习率

def run_epoch(loader,training):  # 运行一轮训练或验证
    model.train(training); losses=[]; positive_dices=[]  # 切换模式并准备记录
    for x,y,_ in loader:  # 逐批读取数据
        x=x.to(DEVICE); y=y.to(DEVICE)  # 移动到运行设备
# ===== 请在此处完成：项目01·任务4 清空训练梯度（开始） =====
        if training:  # 只在训练时更新参数
            # TODO 4：清梯度
            pass  # 等待填写
# ===== 请在此处完成：项目01·任务4 清空训练梯度（结束） =====
        logits=model(x); prob=torch.sigmoid(logits)  # 计算 logits 和概率
        soft_dice=soft_dice_score(prob,y)  # 计算每张切片的软 Dice
        loss=.35*bce(logits,y)+.65*(1-soft_dice).mean()  # 组合加权 BCE 与软 Dice
# ===== 请在此处完成：项目01·任务4 反向传播与更新（开始） =====
        if training:  # 只在训练时更新参数
            # TODO 4：反向传播、梯度裁剪和参数更新
            pass  # 等待填写
# ===== 请在此处完成：项目01·任务4 反向传播与更新（结束） =====
        losses.append(float(loss.detach().cpu()))  # 保存损失
        positive=y.sum((1,2,3))>0  # 找到阳性切片
        if positive.any(): positive_dices.extend(soft_dice[positive].detach().cpu().numpy())  # 单独记录阳性软 Dice
    return float(np.mean(losses)),float(np.mean(positive_dices))  # 返回平均结果

history=[]; best=None; best_d=-1  # 准备保存训练历史和最佳模型
for epoch in range(14):  # 固定训练 14 轮
    tl,td=run_epoch(train_loader,True); vl,vd=run_epoch(val_loader,False)  # 训练并验证
    history.append((tl,td,vl,vd)); print(epoch+1,history[-1])  # 记录当前 epoch 结果
    if vd>best_d: best_d=vd; best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}  # 只按验证阳性软 Dice 选模型
    scheduler.step()  # 更新下一轮学习率
model.load_state_dict(best)  # 载入最佳验证模型


In [ ]:
# 输入：history 中每轮的 loss 和 Dice；输出：task1_training_curve.png。
h=np.array(history)  # 保存当前步骤使用的中间结果
fig,ax=plt.subplots(1,2,figsize=(9,3.5))  # 绘制当前步骤的结果图
ax[0].plot(h[:,0],label='train'); ax[0].plot(h[:,2],label='validation'); ax[0].set_title('loss'); ax[0].legend()  # 计算训练目标并传递梯度
ax[1].plot(h[:,1],label='train'); ax[1].plot(h[:,3],label='validation'); ax[1].set_title('Dice'); ax[1].legend()  # 计算用于比较的评价指标
plt.tight_layout(); plt.savefig(OUT/'task1_training_curve.png',dpi=160); plt.show()  # 绘制当前步骤的结果图


## 任务 5：验证集阈值与测试评价

模型输出概率图。验证集比较 0.25—0.85 的候选阈值，并把阳性切片平均 Dice 最高的阈值固定下来。测试阶段同时对原图和水平翻转图推理，翻回后平均概率；测试集不能参与阈值选择。

**输入与输出：** 输入是最佳验证模型、`val_loader` 和候选阈值；输出是 `val_dices`、`best_threshold`、测试 Dice/IoU、像素指标、空 mask 统计和阳性预测示例。


In [ ]:
# 输入：model、val_loader 和候选 thresholds；输出：val_dices、best_threshold。
thresholds=[round(float(value),2) for value in np.arange(.25,.86,.05)]  # 只在验证集比较候选阈值
val_dices={}  # 保存阈值和阳性切片 Dice

def evaluate(loader,threshold):  # 使用固定阈值评价一个数据集
    model.eval(); ds=[]; ious=[]; positive_ds=[]; positive_ious=[]; examples=[]  # 准备结果
    empty_true=empty_pred=both_empty=0; tp=fp=fn=0  # 准备空 mask 和像素统计
    with torch.no_grad():  # 关闭梯度
        for x,y,pid in loader:  # 逐批读取数据
            x_device=x.to(DEVICE)  # 移动 MRI 到设备
            direct=torch.sigmoid(model(x_device))  # 原方向预测
            flipped=torch.flip(torch.sigmoid(model(torch.flip(x_device,dims=[3]))),dims=[3])  # 水平翻转预测后翻回
            prob=((direct+flipped)/2).cpu(); pred=(prob>=threshold).float()  # 平均两次概率并阈值化
            target_area=y.sum((1,2,3)); pred_area=pred.sum((1,2,3))  # 计算每张切片面积
            empty_true+=int((target_area==0).sum()); empty_pred+=int((pred_area==0).sum())  # 统计空 mask
            both_empty+=int(((target_area==0)&(pred_area==0)).sum())  # 统计共同为空
            inter=(pred*y).sum((1,2,3)); union=((pred+y)>0).float().sum((1,2,3))  # 计算交集和并集
            d=(2*inter+1e-6)/(pred.sum((1,2,3))+y.sum((1,2,3))+1e-6)  # 逐切片 Dice
            i=(inter+1e-6)/(union+1e-6)  # 逐切片 IoU
            positive=target_area>0  # 找到真实阳性切片
            ds.extend(d.numpy()); ious.extend(i.numpy()); positive_ds.extend(d[positive].numpy()); positive_ious.extend(i[positive].numpy())  # 汇总指标
            tp+=int((pred*y).sum()); fp+=int((pred*(1-y)).sum()); fn+=int(((1-pred)*y).sum())  # 汇总像素 TP/FP/FN
            if len(examples)<4:  # 最多保存四张阳性示例
                for k in torch.where(positive)[0].tolist():
                    if len(examples)>=4: break
                    examples.append((x[k,1].numpy(),y[k,0].numpy(),prob[k,0].numpy()))  # 显示 FLAIR 通道、真值和概率
    stats={'total':len(ds),'empty_true':empty_true,'empty_pred':empty_pred,'both_empty':both_empty,'positive_true':len(positive_ds),'positive_dice':float(np.mean(positive_ds)) if positive_ds else None,'positive_iou':float(np.mean(positive_ious)) if positive_ious else None,'pixel_dice':float((2*tp+1e-6)/(2*tp+fp+fn+1e-6)),'pixel_iou':float((tp+1e-6)/(tp+fp+fn+1e-6)),'pixel_precision':float(tp/max(1,tp+fp)),'pixel_recall':float(tp/max(1,tp+fn))}  # 保存结果
    return float(np.mean(ds)),float(np.mean(ious)),examples,stats  # 返回总体和阳性统计

# ===== 请在此处完成：项目01·任务5 验证集阈值选择（开始） =====
# TODO 5：遍历 thresholds，把验证集阳性切片 Dice 写入 val_dices，再选择最高者。
best_threshold=None  # 等待填写
# ===== 请在此处完成：项目01·任务5 验证集阈值选择（结束） =====


## 测试集评价与结果保存

阈值确定后，下面的 `evaluate` 才读取测试集。它返回总体 Dice/IoU、阳性切片指标、像素 TP/FP/FN 指标、可视化样本和空 mask 统计。

**输出：** `task1_prediction.png` 和 `task1_result.json`。图像用于观察边界偏差，JSON 保存本次真实运行的数据规模、患者级划分、阈值和指标。


In [ ]:
# evaluate 已在上一个代码单元格提供；这里使用选出的阈值评价测试集。

test_dice,test_iou,examples,test_stats=evaluate(test_loader,best_threshold)  # 计算用于比较的评价指标
print(best_threshold,test_dice,test_iou)  # 计算用于比较的评价指标


In [ ]:
# 输入：examples、best_threshold 和测试指标；输出：预测图和 task1_result.json。
fig,ax=plt.subplots(len(examples),3,figsize=(8,2.5*len(examples)))  # 建立预测图
for r,(x,y,p) in enumerate(examples):  # 逐张显示
    ax[r,0].imshow(x,cmap='gray'); ax[r,0].set_title('FLAIR')  # 显示 FLAIR
    ax[r,1].imshow(y,cmap='gray'); ax[r,1].set_title('true mask')  # 显示真实 mask
    ax[r,2].imshow(x,cmap='gray'); ax[r,2].imshow(p>=best_threshold,alpha=.4,cmap='viridis'); ax[r,2].set_title('prediction')  # 叠加预测
    for c in range(3): ax[r,c].axis('off')  # 隐藏坐标轴
plt.tight_layout(); plt.savefig(OUT/'task1_prediction.png',dpi=160); plt.show()  # 保存预测图

result={'paired_slices':len(pairs),'patients':len(set(groups)),'train_slices':len(train_ds),'validation_slices':len(val_ds),'test_slices':len(test_ds),'train_patients':len(set(groups[tr_idx])),'validation_patients':len(set(groups[va_idx])),'test_patients':len(set(groups[test_idx])),'input':'source RGB TIFF three registered MRI channels','training_samples_per_epoch':1200,'epochs':14,'best_threshold':best_threshold,'validation_thresholds':val_dices,'test_dice':test_dice,'test_iou':test_iou,'test_mask_stats':test_stats,'seed':SEED,'boundary':'patient-level held-out test; educational small U-Net, not external or clinical validation'}  # 汇总真实运行结果
(OUT/'task1_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')  # 保存结果 JSON
result  # 显示结果


## 结果说明

运行完整 Notebook 后，请记录患者级划分、验证集所选阈值、阳性切片 Dice/IoU、像素指标和空 mask 误报数，并结合 FLAIR、人工 mask 与预测叠加图说明主要边界误差。题目版不预先给出参考数值；需要时再打开实践参考答案核对。
